In [1]:
import numpy as np

In [2]:

def transition(state, action, layout, circle):

    def roll_security_dice():
        trap_triggered = False
        dice_roll = np.random.choice([0, 1])
        return (dice_roll, trap_triggered)

    def roll_normal_dice():
        trap_triggered = np.random.choice([True, False])
        dice_roll = np.random.choice([0, 1, 2])
        return (dice_roll, trap_triggered) 


    def roll_risky_dice():
        trap_triggered = True
        dice_roll = np.random.choice([0, 1, 2, 3])
        return (dice_roll, trap_triggered)
    

    roll_dice_functions = {0:roll_security_dice, 1:roll_normal_dice, 2:roll_risky_dice} 


    (current_position, current_skip_next_turn) = state
    new_skip_next_turn = False

    if current_position == 14:
        print("you won, idiot")
        return (current_position, new_skip_next_turn)
    
    if current_skip_next_turn:
        return (current_position, new_skip_next_turn)
    

    # roll dices
    roll_function = roll_dice_functions[action]
    (dice_roll, trap_triggered) =  roll_function()

    # find new position
    if dice_roll==0:
        new_position = current_position
    else: # if dice roll not 0
        if (current_position == 2):
                if np.random.choice([True, False]):
                    new_position = current_position + dice_roll
                else:
                    new_position = 9+dice_roll

        elif  current_position in range(10): # but not 2

            new_position = current_position+dice_roll

            if new_position > 10:
                if circle:
                    new_position -= 10
                else:
                    new_position = 14

        elif current_position in range(10, 14):
            new_position = current_position+dice_roll

            if new_position > 14:
                if circle:
                    new_position -= 14
                else:
                    new_position = 14

    # Deal with the traps
    if trap_triggered:

        trap = layout[new_position]

        if trap == 4:
            trap = np.random.choice([1, 2, 3])

        if   trap == 1:
            new_position = 0

        elif trap == 2:
            if new_position in range(10, 13):
                new_position -= 7 # -7 -3 = -10
            new_position = max(0, new_position - 3)

        elif trap == 3:
            new_skip_next_turn=True

    return (new_position, new_skip_next_turn)
    

        

In [3]:
layout = np.zeros(15)
circle = 1
state = (0, 0)
action = 2
state


(0, 0)

In [4]:
state=transition(state, action, layout, True)
state

(2, False)

In [31]:

def expected_policy_value(state, action, V, layout, circle, alpha):

    def rolls_security_dice():
        possible_trap_triggered = [False]
        possible_dice_rolls = [0, 1]
        p = 1/2
        return [(p, trap_triggered, dice_roll) for trap_triggered in possible_trap_triggered for dice_roll in possible_dice_rolls]    

    def rolls_normal_dice():
        possible_trap_triggered = [True, False]
        possible_dice_rolls = [0, 1, 2]
        p = 1/6
        return [(p, trap_triggered, dice_roll) for trap_triggered in possible_trap_triggered for dice_roll in possible_dice_rolls]    


    def rolls_risky_dice():
        possible_trap_triggered = [True]
        possible_dice_rolls = [0, 1, 2, 3]
        p = 1/4
        return [(p, trap_triggered, dice_roll) for trap_triggered in possible_trap_triggered for dice_roll in possible_dice_rolls]    

    rolls_dice_functions = {0:rolls_security_dice, 1:rolls_normal_dice, 2:rolls_risky_dice} 



    Q = 0
    (current_position, current_skip_next_turn) = state

    if current_position == 14:
        new_state = (14, False)
        Q +=  0 + alpha*V[new_state]
        return Q 
    
    if current_skip_next_turn:
        new_state = (current_position, False)
        Q += 1 + alpha*V[new_state]
        return Q
    


    # roll dices
    roll_function = rolls_dice_functions[action]
    list_rolls =  roll_function()

    list_new_positions_before_traps = []

    for (p, trap_triggered, dice_roll) in list_rolls:
        # find new position
        if dice_roll==0:
            new_position_before_trap = current_position
            list_new_positions_before_traps.append((p, new_position_before_trap, trap_triggered))
        else: # dice roll is not 0
            if (current_position == 2):
                    new_position_before_trap = current_position + dice_roll
                    list_new_positions_before_traps.append((p*1/2, new_position_before_trap, trap_triggered))
                    new_position_before_trap = 9+dice_roll
                    list_new_positions_before_traps.append((p*1/2, new_position_before_trap, trap_triggered))


            elif  current_position in range(10): # but not 2
                new_position_before_trap = current_position+dice_roll
                if new_position_before_trap > 10:
                    if circle:
                        new_position_before_trap -= 10
                    else:
                        new_position_before_trap = 14
                list_new_positions_before_traps.append((p, new_position_before_trap, trap_triggered))

            elif current_position in range(10, 14):
                new_position_before_trap = current_position+dice_roll
                if new_position_before_trap > 14:
                    if circle:
                        new_position_before_trap -=14
                    else:
                        new_position_before_trap = 14
                list_new_positions_before_traps.append((p, new_position_before_trap, trap_triggered))

    list_new_positions_after_traps = []

    for (p, new_position_before_trap, trap_triggered) in list_new_positions_before_traps:
        # Deal with the traps
        trap = layout[new_position_before_trap]

        if  (not trap_triggered) or (trap == 0):
            new_skip_next_turn = False
            new_position_after_trap = new_position_before_trap
            list_new_positions_after_traps.append((p, (new_position_after_trap, new_skip_next_turn)))
        else:
            trap_list = []

            if trap == 4:
                trap_list.append(1)
                trap_list.append(2)
                trap_list.append(3)
                p/=3
            else:
                trap_list.append(trap)
            
            for trap in trap_list:
                if   trap == 1:
                    new_position_after_trap = 0
                    new_skip_next_turn = False

                elif trap == 2:
                    new_position_after_trap = new_position_before_trap
                    if new_position_after_trap in range(10, 13):
                        new_position_after_trap -= 7 # -7 -3 = -10
                    new_position_after_trap = max(0, new_position_after_trap - 3)
                    new_skip_next_turn = False


                elif trap == 3:
                    new_position_after_trap = new_position_before_trap
                    new_skip_next_turn=True

                list_new_positions_after_traps.append((p, (new_position_after_trap, new_skip_next_turn)))

    # print(list_new_positions_after_traps)
    for (p, new_state) in list_new_positions_after_traps:
        Q +=  p*(1 + alpha*V[new_state])

    return Q





    
def value_iteration(layout, circle, theta, alpha):

    possible_states = []
    for i in range(len(layout)):
        if layout[i]>=3:
            possible_states.append((i, False))
            possible_states.append((i, True))
        else:
            possible_states.append((i, False))

    V = {}
    PI = {}

    for state in possible_states:
        V[state] = 0
        PI[state] = 0
    
    delta = 2*theta
    while delta>=theta:
        print(delta)
        delta = 0
        for state in possible_states:
            v = V[state]
            minv = np.inf
            for action in range(3):
                Q = expected_policy_value(state, action, V, layout, circle, alpha)
                # print(nv)
                if Q <= minv:
                    minv = Q
                    PI[state] = action
            V[state] = minv
            delta = max(abs(v-V[state]), delta)
    print(delta)
    return V, PI
        


In [36]:
# circle: a boolean variable (type bool), indicating if the player must land exactly on
# the final, goal, square 15 to win (circle = True) or still wins by overstepping the final
# square (circle = False).
circle = True

# layout: a vector of type numpy.ndarray that represents the layout of the game, containing 15 values
#         representing the 15 squares of the Snakes and Ladders game:
# layout[i] = 0 if it is an ordinary square
#           = 1 if it is a “restart” trap (go back to square 1)
#           = 2 if it is a “penalty” trap (go back 3 steps)
#           = 3 if it is a “prison” trap (skip next turn)
#           = 4 if it is a “mystery” trap (random effect among the three previous)
# Note that the first and final squares cannot be trapped.

layout = np.ones(15)*4
layout[0] = 0
layout[14] = 0

value_iteration(layout, circle, 0.01, 1)

0.02
2.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
0.9991319444444446
0.994924286265432
0.9836690350651587
0.9303212256068143
0.768996991721334
0.6195796736389134
0.5567018741347578
0.5071763530855069
0.45424685042143764
0.4019857329047021
0.35294449530730176
0.3082557580299081
0.2681845202602311
0.23257791592978094
0.2011229572396651
0.17346429416630116
0.14924759956829092
0.12813198009423488
0.10979283580938315
0.09391234992828856
0.08013934225029473
0.06818934281612599
0.05785064266527229
0.04895078886595883
0.041331728779795185
0.0348413463981565
0.029334344373740606
0.024675680737068717
0.020743267823984723
0.017429124207254176
0.014639239795052106
0.012292652728298492
0.01032015189986879
0.00866287090498119


({(0, False): 19.514799688613262,
  (1, False): 18.025628277244493,
  (1, True): 19.025628277244493,
  (2, False): 16.036690238815385,
  (2, True): 17.036690238815385,
  (3, False): 20.088385593849548,
  (3, True): 21.088385593849548,
  (4, False): 19.262987163048482,
  (4, True): 20.262987163048482,
  (5, False): 17.999974476688216,
  (5, True): 18.999974476688216,
  (6, False): 15.999994998186777,
  (6, True): 16.99999499818678,
  (7, False): 13.999999158112509,
  (7, True): 14.999999158112509,
  (8, False): 11.999999882041518,
  (8, True): 12.999999882041518,
  (9, False): 9.999999986652092,
  (9, True): 10.999999986652092,
  (10, False): 7.999999998832182,
  (10, True): 8.999999998832182,
  (11, False): 5.999999999926227,
  (11, True): 6.999999999926227,
  (12, False): 3.9999999999970264,
  (12, True): 4.999999999997026,
  (13, False): 1.9999999999999432,
  (13, True): 2.999999999999943,
  (14, False): 0},
 {(0, False): 1,
  (1, False): 0,
  (1, True): 2,
  (2, False): 0,
  (2, Tru

In [7]:

initial_position = 0
skip_next_turn = False
state = (initial_position, skip_next_turn)

In [8]:
state = transition(state, 0, layout, circle)
state

(0, False)

In [9]:
"""
Determines the optimal strategy regarding the choice of dice in the S&L game

Parameters
----------
layout : numpy.ndarray
    Represents the 15 squares of S&L game, each layout[i] represents the type of square:
        0 = ordinary square
        1 = restart trap (go back to square 1)
        2 = penalty trap (go back 3 steps)
        3 = prison trap (skip next turn)
        4 = mystery trap (random trap)
        
circle : bool
    True: player must land exactly on final square to win, otherwise loop back to square 1
    False: player wins as soon as they land on or overstep the final square

Returns
-------
expec : numpy.ndarray
    expected cost for each square (excl. final/goal square)    

dice : numpy.ndarray
    choice of dice for each square (excl. final/goal square)

"""
def markovDecision(layout,circle):
    # Initialize solution arrays
    expec = np.zeros(14) # index 0 = square1; index 13 = square 14
    dice = np.zeros(14) # goal square does not need an optimal action
    
    # Implement Value-iteration algo and return optimal policy
    
    
    return [expec, dice]

